In [27]:
import torch

In [28]:
#our dataset location to read
data_location = r"C:\Users\venuv\Documents\ai developer\public\tinyshakesphere.txt".replace("\\", "/")
data_location

'C:/Users/venuv/Documents/ai developer/public/tinyshakesphere.txt'

In [29]:
with open(data_location, "r") as f:
    text = f.read()
print(text[0:100])


First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [30]:
uniqueChars = sorted(list(set(text)))

In [31]:
vocabSize = len(uniqueChars)

In [32]:
#create a dictionary to map characters to integers and from integers to characters
stoi = {ch:i for i, ch in enumerate(uniqueChars)}
itos = {i:ch for i, ch in enumerate(uniqueChars)}

In [33]:
#convert data into numbers
encoded = [stoi[ch] for ch in text]


In [34]:
data = torch.tensor(encoded, dtype=torch.long)

In [35]:
print(data.shape)

torch.Size([1115394])


In [36]:
n = int(0.9*len(data))
training = data[0:n]
testing = data[n:]


In [37]:
block_size=8
batch_size=4

In [38]:
def get_batch(split):

    data = training if split == 'train' else testing

    ix = torch.randint(
        len(data) - block_size,
        (batch_size,)
    )

    x = torch.stack([
        data[i:i+block_size]
        for i in ix
    ])

    y = torch.stack([
        data[i+1:i+block_size+1]
        for i in ix
    ])

    return x,y

In [39]:
#combine everything in a single class
import torch
import torch.nn as nn
import torch.nn.functional as F

class MiniGPT(nn.Module):

    def __init__(
        self,
        vocabSize,
        embedding_size,
        block_size,
        num_heads
    ):

        super().__init__()

        self.embedding_size = embedding_size
        self.block_size = block_size
        self.num_heads = num_heads
        self.head_size = embedding_size // num_heads

        # token embeddings
        self.tokenEmbedding = nn.Embedding(
            vocabSize,
            embedding_size
        )

        # positional embeddings
        self.positionEmbedding = nn.Embedding(
            block_size,
            embedding_size
        )

        # attention layers
        self.query = nn.Linear(
            embedding_size,
            embedding_size,
            bias=False
        )

        self.key = nn.Linear(
            embedding_size,
            embedding_size,
            bias=False
        )

        self.value = nn.Linear(
            embedding_size,
            embedding_size,
            bias=False
        )

        self.outputProjection = nn.Linear(
            embedding_size,
            embedding_size
        )

        # layer norms
        self.layerNorm1 = nn.LayerNorm(
            embedding_size
        )

        self.layerNorm2 = nn.LayerNorm(
            embedding_size
        )

        # feed forward network
        self.ffn = nn.Sequential(

            nn.Linear(
                embedding_size,
                4 * embedding_size
            ),

            nn.ReLU(),

            nn.Linear(
                4 * embedding_size,
                embedding_size
            )
        )

        # final vocab projection
        self.lm_head = nn.Linear(
            embedding_size,
            vocabSize
        )

    def forward(self, x, targets=None):

        B,T = x.shape

        # token embeddings
        tokenEmbeddings = self.tokenEmbedding(x)

        # positional embeddings
        positions = torch.arange(T)

        positionEmbeddings = self.positionEmbedding(
            positions
        )

        x = tokenEmbeddings + positionEmbeddings

        # QKV
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        # split heads
        Q = Q.view(
            B,T,
            self.num_heads,
            self.head_size
        ).permute(0,2,1,3)

        K = K.view(
            B,T,
            self.num_heads,
            self.head_size
        ).permute(0,2,1,3)

        V = V.view(
            B,T,
            self.num_heads,
            self.head_size
        ).permute(0,2,1,3)

        # attention
        attention = (
            Q @ K.transpose(-2,-1)
        ) / (self.head_size ** 0.5)

        # mask
        mask = torch.tril(
            torch.ones(T,T)
        )

        attention = attention.masked_fill(
            mask == 0,
            float('-inf')
        )

        attention = F.softmax(
            attention,
            dim=-1
        )

        # weighted values
        out = attention @ V

        # merge heads
        out = out.permute(
            0,2,1,3
        ).contiguous().view(
            B,T,self.embedding_size
        )

        out = self.outputProjection(out)

        # residual + norm
        x = self.layerNorm1(x + out)

        # ffn
        ffnOut = self.ffn(x)

        # residual + norm
        x = self.layerNorm2(x + ffnOut)

        # logits
        logits = self.lm_head(x)

        loss = None

        if targets is not None:

            B,T,C = logits.shape

            logits = logits.view(
                B*T,
                C
            )

            targets = targets.view(B*T)

            loss = F.cross_entropy(
                logits,
                targets
            )

        return logits, loss
    
    def generate(
    self,
    x,
    max_new_tokens,
    Temperature
    ):

        for _ in range(max_new_tokens):

            # keep only recent context
            x_cond = x[:, -self.block_size:]
            print(x_cond)

            # forward pass
            logits, loss = self(
                x_cond
            )

            # take last token predictions
            logits = logits[:, -1, :]

            #use temperature to control response randomness
            logits/=Temperature

            # convert to probabilities
            probabilities = F.softmax(
                logits,
                dim=-1
            )

            # sample next token
            nextToken = torch.multinomial(
                probabilities,
                num_samples=1
            )

            # append token
            x = torch.cat(
                (x, nextToken),
                dim=1
            )

        return x

In [40]:
model = MiniGPT(
    vocabSize=vocabSize,
    embedding_size=32,
    block_size=8,
    num_heads=4
)

In [41]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3
)

In [42]:
model.parameters

<bound method Module.parameters of MiniGPT(
  (tokenEmbedding): Embedding(65, 32)
  (positionEmbedding): Embedding(8, 32)
  (query): Linear(in_features=32, out_features=32, bias=False)
  (key): Linear(in_features=32, out_features=32, bias=False)
  (value): Linear(in_features=32, out_features=32, bias=False)
  (outputProjection): Linear(in_features=32, out_features=32, bias=True)
  (layerNorm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
  (layerNorm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
  (ffn): Sequential(
    (0): Linear(in_features=32, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=32, bias=True)
  )
  (lm_head): Linear(in_features=32, out_features=65, bias=True)
)>

In [43]:
@torch.no_grad()
def estimate_loss():

    out = {}

    model.eval()

    for split in ['train', 'test']:

        losses = torch.zeros(50)

        for k in range(50):

            x,y = get_batch(split)

            logits, loss = model(x,y)

            losses[k] = loss.item()

        out[split] = losses.mean()

    model.train()

    return out

In [44]:
for step in range(5000):

    if step % 500 == 0:

        losses = estimate_loss()

        print(
            "step:",
            step,
            "train loss:",
            losses['train'].item(),
            "test loss:",
            losses['test'].item()
        )

    x,y = get_batch('train')

    logits, loss = model(x,y)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

step: 0 train loss: 4.304589748382568 test loss: 4.316982269287109
step: 500 train loss: 2.7883005142211914 test loss: 2.718156337738037
step: 1000 train loss: 2.579702138900757 test loss: 2.560800790786743
step: 1500 train loss: 2.4695568084716797 test loss: 2.516627073287964
step: 2000 train loss: 2.4200563430786133 test loss: 2.473923683166504
step: 2500 train loss: 2.397707939147949 test loss: 2.3982534408569336
step: 3000 train loss: 2.3971567153930664 test loss: 2.3376922607421875
step: 3500 train loss: 2.3531837463378906 test loss: 2.423551082611084
step: 4000 train loss: 2.3676867485046387 test loss: 2.357663631439209
step: 4500 train loss: 2.317166328430176 test loss: 2.3548460006713867


In [45]:
context = torch.zeros(
    (1,1),
    dtype=torch.long
)

In [46]:
context

tensor([[0]])

In [53]:
generated = model.generate(
    context,
    max_new_tokens=100,
    Temperature=0.7
)

tensor([[0]])
tensor([[ 0, 18]])
tensor([[ 0, 18, 47]])
tensor([[ 0, 18, 47, 50]])
tensor([[ 0, 18, 47, 50, 50]])
tensor([[ 0, 18, 47, 50, 50,  1]])
tensor([[ 0, 18, 47, 50, 50,  1, 39]])
tensor([[ 0, 18, 47, 50, 50,  1, 39, 57]])
tensor([[18, 47, 50, 50,  1, 39, 57,  6]])
tensor([[47, 50, 50,  1, 39, 57,  6,  0]])
tensor([[50, 50,  1, 39, 57,  6,  0, 20]])
tensor([[50,  1, 39, 57,  6,  0, 20, 53]])
tensor([[ 1, 39, 57,  6,  0, 20, 53, 61]])
tensor([[39, 57,  6,  0, 20, 53, 61, 11]])
tensor([[57,  6,  0, 20, 53, 61, 11,  0]])
tensor([[ 6,  0, 20, 53, 61, 11,  0, 18]])
tensor([[ 0, 20, 53, 61, 11,  0, 18, 39]])
tensor([[20, 53, 61, 11,  0, 18, 39, 56]])
tensor([[53, 61, 11,  0, 18, 39, 56, 42]])
tensor([[61, 11,  0, 18, 39, 56, 42, 39]])
tensor([[11,  0, 18, 39, 56, 42, 39, 51]])
tensor([[ 0, 18, 39, 56, 42, 39, 51, 39]])
tensor([[18, 39, 56, 42, 39, 51, 39, 42]])
tensor([[39, 56, 42, 39, 51, 39, 42,  1]])
tensor([[56, 42, 39, 51, 39, 42,  1, 60]])
tensor([[42, 39, 51, 39, 42,  1, 60, 4

In [54]:
generated

tensor([[ 0, 18, 47, 50, 50,  1, 39, 57,  6,  0, 20, 53, 61, 11,  0, 18, 39, 56,
         42, 39, 51, 39, 42,  1, 60, 43,  1, 61, 39, 58, 46,  1, 46, 53, 59, 49,
          1, 45, 56, 53, 60, 43,  1, 58, 53,  1, 57, 53, 52, 42,  1, 57, 39, 47,
         52, 45, 43, 57,  1, 63, 53, 59,  1, 46, 53, 59, 52, 42,  1, 59, 56, 42,
         63,  6,  0, 17, 52, 42, 10,  0, 35, 46, 53, 59,  1, 42, 43, 56,  1, 58,
         46, 47, 57,  1, 47, 52, 45, 43, 57,  1, 39]])

In [55]:
decoded = ''.join(
    itos[i]
    for i in generated[0].tolist()
)

print(decoded)


Fill as,
How;
Fardamad ve wath houk grove to sond sainges you hound urdy,
End:
Whou der this inges a
